# Task 0: Setting up Ollama

In [108]:
!pip install openai
!ollama pull gemma3:1b
!ollama pull gemma3:4b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 7cd4618c1faf: 100% ▕██████████████████▏ 815 MB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084c7d92a3: 100% ▕██████████████████▏ 8.4 KB                         
pulling 3116c5225075: 100% ▕██████████████████▏   77 B                         
pulling 120007c81bf8: 100% ▕██████████████████▏  492 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling aeda25e63ebd: 100% ▕██████████████████▏ 3.3 GB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084

In [109]:
# a) Connecting to Ollama via OpenAI's API
from openai import OpenAI
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [110]:
# b) Test gemma3:1b
from IPython.display import display, Markdown
response = client.chat.completions.create(
    model="gemma3:1b",
    messages=[{"role": "user", "content": "Wassup"}],
    max_tokens=100
)
print(response.choices[0].message.content)

Not much, just hanging out! How about you? What’s up with you? 😊


In [111]:
# b) Test gemma3:1b
response = client.chat.completions.create(
    model="gemma3:1b",
    messages=[{"role": "user", "content": "tell a joke"}],
    max_tokens=100
)
display(Markdown(response.choices[0].message.content))

Why did the scarecrow win an award? 

Because he was outstanding in his field! 😊 

---

Want to hear another one?

In [112]:
# Part c: Test gemma3:4b
response = client.chat.completions.create(
    model="gemma3:4b",
    messages=[{"role": "user", "content": "Wassup"}]
)
display(Markdown(response.choices[0].message.content))

Not much, just hanging out and ready to chat! 😄 

What's up with you? How's your day going? 

Is there anything you'd like to talk about, or were you just saying hello?

In [113]:
response = client.chat.completions.create(
    model="gemma3:4b",
    messages=[{"role": "user", "content": "What game do you have"}]
)
display(Markdown(response.choices[0].message.content))

As a large language model, I don't "have" a game in the way a person does. I don't experience play or competition. 

However, you could say I'm constantly engaged in a kind of game – a continuous process of **learning and responding to your prompts.** 

Here's how it plays out:

* **You give me a prompt:** This is like the starting point of a game.
* **I analyze your prompt:** I try to understand what you're asking for.
* **I generate a response:** This is my "move" – crafting an answer based on my training data.
* **You evaluate my response:** You assess whether my answer was helpful, accurate, and relevant.

**Think of it like a conversation.**  Each turn is a new stage in the game. My goal is to play that game well – to provide you with the best possible answer or completion.

**I'm essentially a very advanced conversational AI, continuously adapting and improving based on our interactions!**

**Do you want to play a game?** I can try:

*   **Trivia:** I can test your knowledge on various topics.
*   **Storytelling:**  We can create a story together, with you and me taking turns adding to it.
*   **Word Games:**  Like "20 Questions" or "Hangman."


Let me know what you'd like to do!

# Task 1: Text classification with Ollama

In [114]:
import pandas as pd
emails = pd.read_csv("emails.csv", sep=";")
print(emails.head())

                                            headline
0  URGENT: Your account will be suspended within ...
1  Congratulations! You have won a 1000€ gift car...
2  Hot singles in your area are waiting to meet y...
3  Re: Inheritance transfer of 4.5M USD pending y...
4       Meeting agenda for Thursday's project review


In [115]:
# a) Make a function for classifying emails 
def classify_email(headline, model="gemma3:1b"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": f"""Classify the following email headline as exactly one of: spam, work, or unknown.
Reply with only one word: spam, work, or unknown.

Email headline: {headline}"""
            }
        ],
        max_tokens=10
    )
    return response.choices[0].message.content.strip().lower()

In [116]:
# b) Classify emails using gemma3:1b
emails["classification_1b"] = emails["headline"].apply(lambda x: classify_email(x, model="gemma3:1b"))
display(emails[["headline", "classification_1b"]])

,headline,classification_1b
0,URGENT: Your account will be suspended within ...,spam
1,Congratulations! You have won a 1000€ gift car...,spam
2,Hot singles in your area are waiting to meet y...,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,work
4,Meeting agenda for Thursday's project review,work
5,"Q3 budget report attached, please review by Fr...",work
6,Reminder: Annual performance review scheduled ...,work
7,"Updated draft of the manuscript, comments welcome",work
8,Quick question about last week,unknown
9,Following up,unknown


In [117]:
# c) Classify emails using gemma3:4b
emails["classification_4b"] = emails["headline"].apply(lambda x: classify_email(x, model="gemma3:4b"))
display(emails[["headline", "classification_1b", "classification_4b"]])

,headline,classification_1b,classification_4b
0,URGENT: Your account will be suspended within ...,spam,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,work,spam
4,Meeting agenda for Thursday's project review,work,work
5,"Q3 budget report attached, please review by Fr...",work,work
6,Reminder: Annual performance review scheduled ...,work,work
7,"Updated draft of the manuscript, comments welcome",work,work
8,Quick question about last week,unknown,unknown
9,Following up,unknown,work


There are some clear differences in the classifications. It looks like gemma3:4b has done a better job identyfing the correct classification for each email.

In [118]:
import pandas as pd
#d) Write a script that repeats b) and c) 3 times

results_1b = []
results_4b = []

for i in range(3):
    run_1b = emails["headline"].apply(lambda x: classify_email(x, model="gemma3:1b"))
    run_4b = emails["headline"].apply(lambda x: classify_email(x, model="gemma3:4b"))
    results_1b.append(run_1b.values)
    results_4b.append(run_4b.values)

# Build DataFrame
df_results = pd.DataFrame({
    "headline": emails["headline"],
    "1b_run1": results_1b[0],
    "1b_run2": results_1b[1],
    "1b_run3": results_1b[2],
    "4b_run1": results_4b[0],
    "4b_run2": results_4b[1],
    "4b_run3": results_4b[2],
})

display(df_results)

,headline,1b_run1,1b_run2,1b_run3,4b_run1,4b_run2,4b_run3
0,URGENT: Your account will be suspended within ...,spam,spam,spam,spam,spam,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam,spam,spam,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam,spam,spam,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,work,work,work,spam,spam,spam
4,Meeting agenda for Thursday's project review,work,work,work,work,work,work
5,"Q3 budget report attached, please review by Fr...",work,work,work,work,work,work
6,Reminder: Annual performance review scheduled ...,work,work,work,work,work,work
7,"Updated draft of the manuscript, comments welcome",work,work,work,work,work,work
8,Quick question about last week,unknown,unknown,unknown,unknown,unknown,unknown
9,Following up,unknown,unknown,unknown,unknown,work,work


The gemma3:1b output is identical for all three rounds. The gemma3:4b has some variation in line 9, the "Following up" is classified as both work and unknown. Which is better depend on what queality you prioritsie conistency or accuracy / the level of advancement. 

# Task 2: Sentiment analysis with Ollama

In [119]:
news = pd.read_csv("news.csv", sep=";")
print(news.head())

                                            headline
0  Nordion Industries beats Q1 earnings estimates...
1  Helvora Pharmaceuticals misses earnings foreca...
2  Aurelis Bank reports steady quarterly profit, ...
3  Veridyne Logistics to acquire rival Trantec in...
4  Antitrust regulators block proposed merger bet...


In [120]:
# a) Make a function for classifying the texts

import json

def classify_news(headline, model="gemma3:4b"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f'Classify this headline. Reply with only JSON like {{"topic": "earnings", "sentiment": "positive"}}. Topic: earnings/mergers/regulation/macroeconomics. Sentiment: positive/negative/neutral. Headline: {headline}'}],
        max_tokens=50
    )
    result = response.choices[0].message.content.strip().replace("```json","").replace("```","").strip()
    return json.loads(result)

In [121]:
# b) Classify all the news
news["topic"] = news["headline"].apply(lambda x: classify_news(x)["topic"])
news["sentiment"] = news["headline"].apply(lambda x: classify_news(x)["sentiment"])
display(news)

,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,earnings,positive
1,Helvora Pharmaceuticals misses earnings foreca...,earnings,negative
2,"Aurelis Bank reports steady quarterly profit, ...",earnings,positive
3,Veridyne Logistics to acquire rival Trantec in...,mergers,positive
4,Antitrust regulators block proposed merger bet...,regulation,negative
5,Kestrel Semiconductor confirms early-stage mer...,mergers,neutral
6,New EU AI Act compliance rules expected to rai...,regulation,negative
7,Finnish FSA grants Norvik Capital expanded lic...,regulation,positive
8,"Eurozone inflation cools to 2.1%, easing press...",macroeconomics,positive
9,Rising interest rates weigh on Tessaro Real Es...,economics,negative


*c) The same data and prompt by classified by Claude*



| # | Headline | Topic | Sentiment |
|---|----------|-------|-----------|
| 0 | Nordion Industries beats Q1 earnings estimates | earnings | positive |
| 1 | Helvora Pharmaceuticals misses earnings forecast | earnings | negative |
| 2 | Aurelis Bank reports steady quarterly profit | earnings | neutral |
| 3 | Veridyne Logistics to acquire rival Trantec | mergers | neutral |
| 4 | Antitrust regulators block proposed merger | regulation | negative |
| 5 | Kestrel Semiconductor confirms early-stage merger | mergers | neutral |
| 6 | New EU AI Act compliance rules expected to raise costs | regulation | negative |
| 7 | Finnish FSA grants Norvik Capital expanded license | regulation | positive |
| 8 | Eurozone inflation cools to 2.1%, easing pressure | macroeconomics | positive |
| 9 | Rising interest rates weigh on Tessaro Real Estate | macroeconomics | negative |

Some minor differences. Row 9 is classified as macroeconomics by Claude while gemma3;4b classified it as economics. Claude also classified two "positive" news as neutral. Claude seems to have been more accurate.

# Task 3: Supervised machine learning

In [122]:
df = pd.read_csv("bank-additional.csv", sep=";")

In [123]:
# a) Exploratory Data Analysis

# Preview the data 
display(df.head())
print(f"Total rows: {len(df)}")

# Checking for missing values, duplicated rows and data types
print(f"Missing vales: {df.isnull().sum()}")
print(f"Duplicated rows: {df.duplicated().sum()}")
print(df.dtypes)

# Basic stats
print("\nBasic statistics:\n")
display(df.describe())

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


Total rows: 4119
Missing vales: age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64
Duplicated rows: 0
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       floa

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [124]:
# b) Preprocess the data using the appropriate methods

from sklearn.preprocessing import StandardScaler

# Make a copy
df_processed = df.copy()

# Encode target variable y as 0/1
df_processed["y"] = (df_processed["y"] == "yes").astype(int)

# One-hot encode all categorical columns
df_processed = pd.get_dummies(df_processed, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
num_cols = ["age", "duration", "campaign", "pdays", "previous", 
            "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
df_processed[num_cols] = scaler.fit_transform(df_processed[num_cols])

display(df_processed.shape)
display(df_processed.head())

(4119, 54)

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,-0.980752,0.903952,-0.209228,0.201031,-0.351356,-1.206054,-1.185448,-1.240939,-1.331707,-0.914779,...,True,False,False,False,False,False,False,False,True,False
1,-0.107991,0.350300,0.569634,0.201031,-0.351356,0.649441,0.715193,0.892269,0.711698,0.332862,...,True,False,False,False,False,False,False,False,True,False
2,-1.465619,-0.116966,-0.598660,0.201031,-0.351356,0.841389,1.528273,-0.283172,0.773427,0.836535,...,False,False,False,False,False,False,False,True,True,False
3,-0.204965,-0.941553,0.180203,0.201031,-0.351356,0.841389,1.528273,-0.283172,0.771697,0.836535,...,False,False,False,False,False,False,False,False,True,False
4,0.667795,-0.780563,-0.598660,0.201031,-0.351356,-0.118350,-0.655478,-0.326707,0.328632,0.398028,...,False,True,False,False,True,False,False,False,True,False


In [125]:
# c) Part 1: Choose three different machine learning algorithms from scikit-learn 

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# I. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
print("I:", accuracy_score(y_val, lr.predict(X_val)))

# II. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("II:", accuracy_score(y_val, dt.predict(X_val)))

# III. KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
print("III:", accuracy_score(y_val, knn.predict(X_val)))

I: 0.9150485436893204
II: 0.9077669902912622
III: 0.8980582524271845


I chose these machine learning algorithms from scikit-learn because this is a classification task and these are models which work with classifictaion tasks. I like to work with logistics regeression as I am already very familiar with the model, the decision tree is easy to understand and can capture nonlinear relationships, and I wanted to try out the KNN. 

The validation accuracy numbers shows what percentage of the validation set the model predicted right. The results show the highest percenatge score for the logistic regression model.

In [126]:
# c) Part 2: Train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance

# I. Increasing the complexity penalizaton (decreasing C from original model) 
lr2 = LogisticRegression(max_iter=1000, C=0.1)
lr2.fit(X_train, y_train)
print("I(C=0.1):", accuracy_score(y_val, lr2.predict(X_val)))

# II. Making the tree more shallow
dt2 = DecisionTreeClassifier(random_state=42, max_depth=5)
dt2.fit(X_train, y_train)
print("II(max_depth=5):", accuracy_score(y_val, dt2.predict(X_val)))

# III. Increasing the amount of neighbors
knn2 = KNeighborsClassifier(n_neighbors=10)
knn2.fit(X_train, y_train)
print("III(n_neighbors=10):", accuracy_score(y_val, knn2.predict(X_val)))

I(C=0.1): 0.9101941747572816
II(max_depth=5): 0.9223300970873787
III(n_neighbors=10): 0.8956310679611651


In [127]:
# I. Changing to L1 penalty
lr3 = LogisticRegression(max_iter=1000, penalty="l1", solver="liblinear")
lr3.fit(X_train, y_train)
print("I(L1):", accuracy_score(y_val, lr3.predict(X_val)))

# II. Increasing minimum amount of samles per leaf
dt3 = DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_leaf=10)
dt3.fit(X_train, y_train)
print("II(min_leaf=10):", accuracy_score(y_val, dt3.predict(X_val)))

# III. Weighting based on distance
knn3 = KNeighborsClassifier(n_neighbors=5, weights="distance")
knn3.fit(X_train, y_train)
print("III(weights):", accuracy_score(y_val, knn3.predict(X_val)))

I(L1): 0.9174757281553398
II(min_leaf=10): 0.9174757281553398
III(weights): 0.8980582524271845


In [128]:
# The final best versions of each model:

# I. Regressionn model
lr3 = LogisticRegression(max_iter=1000, penalty="l1", solver="liblinear")
lr3.fit(X_train, y_train)
print("I:", accuracy_score(y_val, lr3.predict(X_val)))

# II. Decision tree
dt2 = DecisionTreeClassifier(random_state=42, max_depth=5)
dt2.fit(X_train, y_train)
print("II(max_depth=5):", accuracy_score(y_val, dt2.predict(X_val)))

# III. KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
print("III:", accuracy_score(y_val, knn.predict(X_val)))

I: 0.9174757281553398
II(max_depth=5): 0.9223300970873787
III: 0.8980582524271845


In [129]:
# d) Train, validation and test set split versus cross-validation

from sklearn.model_selection import cross_val_score
import numpy as np

# Cross-validation
print("Cross-validation")
print("I:", np.mean(cross_val_score(lr3, X, y, cv=5)))
print("II:", np.mean(cross_val_score(dt2, X, y, cv=5)))
print("III:", np.mean(cross_val_score(knn, X, y, cv=5)))

# Train, validationa nd test set split

print("\nTrain, validation an test set split")
print("I:", accuracy_score(y_val, lr3.predict(X_val)))
print("II:", accuracy_score(y_val, dt2.predict(X_val)))
print("III:", accuracy_score(y_val, knn.predict(X_val)))

Cross-validation
I: 0.9123574065991106
II: 0.904829300805719
III: 0.9021602826504973

Train, validation an test set split
I: 0.9174757281553398
II: 0.9223300970873787
III: 0.8980582524271845


We can see from the results that the train, validation and test split gave higher accuracy validation scores. However, the cross-validation is more trustworthy because it test the model on different parts of the data and multiple times.

In [130]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score

models = {"Logistic Regression": lr3, "Decision Tree": dt2, "KNN": knn}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name}:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



Logistic Regression:
Accuracy: 0.904126213592233
Precision: 0.6
Recall: 0.42391304347826086
ROC-AUC: 0.9276401758137325
Confusion Matrix:
 [[706  26]
 [ 53  39]]

Decision Tree:
Accuracy: 0.9029126213592233
Precision: 0.5666666666666667
Recall: 0.5543478260869565
ROC-AUC: 0.8229092421002614
Confusion Matrix:
 [[693  39]
 [ 41  51]]

KNN:
Accuracy: 0.8968446601941747
Precision: 0.5714285714285714
Recall: 0.30434782608695654
ROC-AUC: 0.8341054882394868
Confusion Matrix:
 [[711  21]
 [ 64  28]]


The logistics regression seems to be the best model. It scores the higest in accuarcy, precision and ROC-AUC. The confusion matrix shows 39 correctly predicted yes and 706 correctly predicted no.